# 01 - Data Preprocessing

This notebook loads and preprocesses the two cohorts used in this study:
- **TCGA-BRCA** (training): 213 patients with RNA-seq expression and clinical data
- **GSE96058 / SCAN-B** (external validation): 1,483 patients

Key steps:
1. Load raw clinical data
2. Inspect data quality and missing values
3. Construct the binary outcome variable (high_risk)
4. Filter to patients with defined outcomes
5. Summarize cohort characteristics

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data_loader import load_tcga_feature_matrix, load_clinical_data
from src.preprocessing import filter_outcome

## 1. Load TCGA-BRCA Data

In [ ]:
# TCGA feature matrix (already preprocessed with pathway scores)
tcga_fm = load_tcga_feature_matrix('../data/processed/02_tcga_feature_matrix.csv')
print(f"\nTCGA feature matrix shape: {tcga_fm.shape}")
print(f"Columns: {tcga_fm.columns.tolist()}")
tcga_fm.head()

In [ ]:
# TCGA clinical data
tcga_clin = load_clinical_data('../data/clinical/01_tcga_clinical.csv')
print(f"\nTCGA clinical data shape: {tcga_clin.shape}")
print(f"\nMissing values in key columns:")
for col in ['high_risk', 'time_to_event', 'event_status']:
    if col in tcga_clin.columns:
        print(f"  {col}: {tcga_clin[col].isna().sum()} missing")

## 2. Load GSE96058 / SCAN-B Data

In [ ]:
gse_clin = load_clinical_data('../data/clinical/01_gse96058_clinical.csv')
print(f"\nGSE96058 clinical data shape: {gse_clin.shape}")
print(f"\nColumn names:")
print(gse_clin.columns.tolist())

In [ ]:
# Check clinical feature distributions
print("\n=== GSE96058 Clinical Feature Summary ===")
for col in ['er_status', 'pgr_status', 'her2_status', 'ki67_status', 'nhg', 'pam50_subtype', 'lymph_node_status']:
    if col in gse_clin.columns:
        print(f"\n{col}:")
        print(gse_clin[col].value_counts(dropna=False).head(10))

## 3. Outcome Variable

The binary outcome `high_risk` is defined as:
- **high_risk = 1**: Patient died or had disease progression within 5 years (60 months)
- **high_risk = 0**: Patient survived beyond 5 years event-free
- **Excluded**: Censored patients with follow-up < 5 years (outcome unknown)

In [ ]:
# TCGA outcome summary
print("=== TCGA Outcome Distribution ===")
print(f"Total patients in feature matrix: {len(tcga_fm)}")
print(f"high_risk distribution:")
print(tcga_fm['high_risk'].value_counts())
print(f"\nClass balance: {tcga_fm['high_risk'].mean():.1%} high risk")

In [ ]:
# GSE96058 outcome summary
gse_filtered = filter_outcome(gse_clin)
print(f"\n=== GSE96058 Outcome Distribution ===")
print(f"Total patients before filtering: {len(gse_clin)}")
print(f"Patients with defined outcome: {len(gse_filtered)}")
print(f"high_risk distribution:")
print(gse_filtered['high_risk'].value_counts())
print(f"\nClass balance: {gse_filtered['high_risk'].mean():.1%} high risk")

## 4. Summary Statistics

In [ ]:
print("=" * 60)
print("COHORT SUMMARY")
print("=" * 60)
print(f"\nTCGA-BRCA (training):     {len(tcga_fm):>5} patients")
print(f"GSE96058/SCAN-B (valid.): {len(gse_filtered):>5} patients")
print(f"\nFeatures:")
print(f"  7 pathway scores + 1 ratio feature = 8 pathway features")
print(f"  9 clinical features (GSE96058 only)")
print(f"  17 combined features (GSE96058)")